In [1]:
# import libraries
import pandas as pd
import ee
import geemap

## Connect to Google Earth Engine (GEE)

In [ ]:
# Authenticate GEE
ee.Authenticate()

# Initialize GEE
EE_PROJECT_ID = ""   # Change to your project ID

#ee.Initialize(project=EE_PROJECT_ID)
ee.Initialize()

## . Visualization Parameters

In [3]:
# Center coordinates to show map
fct_center =  (9.056266, 7.498522)

In [ ]:
# Boundary visualization params 
vis_params_fao_1 = {
  "fillColor": 'b5ffb4',
  "color": '00909F',
  "width": 1.0,
}

vis_params_aoi = {"fillcolor": "", "color": "red"}

# Sentinel-2 Visualization parameters 
vis_params_s2_rgb = {"min" : 300, "max" :3000, "bands": ["B4", "B3", "B2"]}
vis_params_s2_fcc = {"min" : 300, "max" :3000, "bands": ["B8", "B4", "B3"]}


# Spectral indices visualization parameters
# NDVI
ndvi_vis = {
    "min": -0.2,
    "max": 0.8,
    "palette": [
        "#a50026",  
        "#d73027",
        "#f46d43",
        "#fdae61",
        "#fee08b",
        "#d9ef8b",
        "#a6d96a",
        "#66bd63",
        "#1a9850",
        "#006837",  
    ],
}

# NDBI
ndbi_vis = {
    "min": -0.5,
    "max": 0.5,
    "palette": [
        "#f7f7f7",  
        "#fddbc7",
        "#f4a582",
        "#d6604d",
        "#b2182b",  
    ],
}

# NDWI
ndwi_vis = {
    "min": -0.5,
    "max": 0.5,
    "palette": [
        "#543005",  
        "#8c510a",
        "#d8b365",
        "#f6e8c3",
        "#c7eae5",
        "#5ab4ac",
        "#01665e",  
    ],
}


# Visualisation parameters for road layers
vis_params_roads_vector = {
                        "color": "red",
                        "width": 1.5,
                    }

vis_params_roads_raster = {
                        "min": 0,
                        "max": 1,
                        "palette": ["black", "white"],
                    }


vis_params_dist_road = {
    "min": 0,
    "max": 21730, #meters
    "palette": [
        "#d73027",  
        "#f46d43",
        "#fdae61",
        "#fee08b",
        "#d9ef8b",
        "#a6d96a",
        "#1a9850",  
    ],
}

# Visualisation parameters for water layers
vis_params_water = {
    "min": 0,
    "max": 1,
    "palette": ["white", "blue"], 
}

# Distance to water
vis_params_dist_water = {
    "min": 0,
    "max": 38497,  
    "palette": [
        "#f7fbff",  
        "#deebf7",
        "#9ecae1",
        "#4292c6",
        "#2171b5",
        "#08306b",  
    ],
}


# Nighttime lights visualisation parameters
vis_params_ntl = {
    "min": 0,
    "max": 20,  
    "palette": [
        "#000000",  
        "#2c0b00",
        "#6e1c00",
        "#a83800",
        "#d9720a",
        "#f7b733",
        "#ffe98a",  
    ],
}


# Visualization parameters for GPWv411 population DENSITY layer
vis_params_gpw = {
  "min": 0.0,
  "max": 10000.0,
  "palette": ["ffffe7", "FFc869", "ffac1d", "e17735", "f2552c", "9f0c21"]
}

## Boundary Data

In [5]:
# Boundary data from FAO GAUL
# Data source: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_SIMPLIFIED_500m_2015_level0

fao_gaul_l0 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level0') # Countries boundaries
fao_gaul_l1 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level1') # States boundaries
fao_gaul_l2 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level1') # LGAs boundaries



# Create map to visualize the FAO GAUL boundary data
boundary_map = geemap.Map(center=(7.0, 8.0), zoom=10)

boundary_map.addLayer(fao_gaul_l0, {}, 'Country Boundaries')
boundary_map.addLayer(fao_gaul_l1, {}, 'State Boundaries')
boundary_map.addLayer(fao_gaul_l2, {}, 'LGA Boundaries')
boundary_map

Map(center=[7.0, 8.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

In [6]:
# Check columns
print(fao_gaul_l0.limit(0).getInfo()["columns"])

# Extract Nigerian boundaries from FAO GAUL
nga_l0 = fao_gaul_l0.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
nga_l1 = fao_gaul_l1.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
fct_l0 = nga_l1.filter(ee.Filter.eq("ADM1_NAME", "Abuja")) # FCT extent
print(fct_l0.getInfo())

# Get geometry from FCT boundary (FeatureCollection)
aoi = fct_l0.geometry()
aoi_bbox = aoi.bounds()

# Creat map to visualize FCT boundary data
aoi_map = geemap.Map(center=fct_center, zoom=10)
aoi_map.addLayer(fct_l0, vis_params_aoi, 'FCT Boundary')
aoi_map

{'ADM0_CODE': 'Integer', 'ADM0_NAME': 'String', 'DISP_AREA': 'String', 'EXP0_YEAR': 'Integer', 'STATUS': 'String', 'STR0_YEAR': 'Integer', 'Shape_Area': 'Float', 'Shape_Leng': 'Float', 'system:index': 'String'}
{'type': 'FeatureCollection', 'columns': {'ADM0_CODE': 'Integer', 'ADM0_NAME': 'String', 'ADM1_CODE': 'Integer', 'ADM1_NAME': 'String', 'DISP_AREA': 'String', 'EXP1_YEAR': 'Integer', 'STATUS': 'String', 'STR1_YEAR': 'Integer', 'Shape_Area': 'Float', 'Shape_Leng': 'Float', 'system:index': 'String'}, 'version': 1701682755394127, 'id': 'FAO/GAUL_SIMPLIFIED_500m/2015/level1', 'properties': {'system:asset_size': 80042928}, 'features': [{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[6.746250262158974, 8.996497434811962], [6.748495969531178, 8.992005993451773], [6.748495969531178, 8.989760187648406], [6.748495969531178, 8.985268685352079], [6.748495969531178, 8.983022905093227], [6.748495969531178, 8.98077718923787], [6.750741754879445, 8.971794157254672], [6.757

Map(center=[9.056266, 7.498522], controls=(WidgetControl(options=['position', 'transparent_bg'], position='top…

## Explore Image operations

In [7]:
# Image Collection
s2_img_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') # All Sentinel-2 collection
    .filterDate('2022-01-01', '2022-01-31') # Limit/filter to January 2022
    .filterBounds(fct_l0.geometry()) # Limit/filter to Abuja
    # Pre-filter to get less cloudy granules.
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

# Collection Properties
print(f"Number of images in S2 collection: {s2_img_col.size().getInfo()}\n")

# First image in collection
first_s2_img = s2_img_col.first()
print(f"First image in S2 collection: {first_s2_img.getInfo()}\n")

# Bands in S2 image
print(first_s2_img.bandNames().getInfo())

# Select just a few bands
red_band_s2 = first_s2_img.select(["B4", "B3", "B2"])
print(f"Red Band: {red_band_s2.bandNames().getInfo()}")


# Visualize S2 image
s2_map = geemap.Map(center=fct_center, zoom=8)
s2_map.addLayer(first_s2_img.clip(aoi), vis_params_s2_rgb, "First S2 Img")
s2_map

Number of images in S2 collection: 20

First image in S2 collection: {'type': 'Image', 'bands': [{'id': 'B1', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [1830, 1830], 'crs': 'EPSG:32632', 'crs_transform': [60, 0, 199980, 0, -60, 1000020]}, {'id': 'B2', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [10980, 10980], 'crs': 'EPSG:32632', 'crs_transform': [10, 0, 199980, 0, -10, 1000020]}, {'id': 'B3', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [10980, 10980], 'crs': 'EPSG:32632', 'crs_transform': [10, 0, 199980, 0, -10, 1000020]}, {'id': 'B4', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [10980, 10980], 'crs': 'EPSG:32632', 'crs_transform': [10, 0, 199980, 0, -10, 1000020]}, {'id': 'B5', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [5490, 5490], 

Map(center=[9.056266, 7.498522], controls=(WidgetControl(options=['position', 'transparent_bg'], position='top…

In [8]:
# Mosaic the entire S2 collection & clip to FCT extent [Spatial Mosaic]
s2_mosaic = s2_img_col.mosaic()
s2_mosaic_clipped = s2_mosaic.clip(aoi)
print(f"Mosaiced S2 Collection {s2_mosaic.getInfo()}")

s2_map.addLayer(s2_mosaic_clipped, vis_params_s2_rgb, "S2 Mosaiced - Spatial")
s2_map

Mosaiced S2 Collection {'type': 'Image', 'bands': [{'id': 'B1', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B2', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B3', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B4', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B5', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B6', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B7', 'data_type': {'type': 'PixelTy

Map(center=[9.056266, 7.498522], controls=(WidgetControl(options=['position', 'transparent_bg'], position='top…

In [9]:
# Compute median composite the entire S2 collection & clip to FCT extent [Temporal Composite]
s2_median = s2_img_col.median().clip(aoi)
print(f"Median of S2 collection: {s2_median.getInfo()}")

s2_map.addLayer(s2_median, vis_params_s2_rgb, "S2 Median")
s2_map

Median of S2 collection: {'type': 'Image', 'bands': [{'id': 'B1', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'dimensions': [3, 3], 'origin': [6, 7], 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B2', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'dimensions': [3, 3], 'origin': [6, 7], 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B3', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'dimensions': [3, 3], 'origin': [6, 7], 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B4', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'dimensions': [3, 3], 'origin': [6, 7], 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B5', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'dimensions': [3, 3], 'origin': [6, 7], 'crs': 'EPSG:4326', 'crs_transform'

Map(center=[9.056266, 7.498522], controls=(WidgetControl(options=['position', 'transparent_bg'], position='top…

In [10]:
# Compute Spectral indices


## . Land Cover Data Download

## Drivers of Urban Growth / Predictors

**Elevation/DEM**

In [39]:
# Download elevation and compute slope
dem = ee.Image('USGS/SRTMGL1_003')
dem.bandNames().getInfo()



elevation = dem.select('elevation')
slope = ee.Terrain.slope(elevation)


dem_map = geemap.Map(center = [7.08, 8.88], zoom=8)
dem_map.add_basemap("SATELLITE")
dem_map.addLayer(dem.clip(aoi), {'min': 0, 'max': 500, "palette": ["Red", "Green", "Yellow"]}, "DEM")
dem_map.addLayer(slope.clip(aoi), {'min': 0, 'max': 10, "palette": ["Red", "Green", "Yellow"]}, "Slope")
dem_map


Map(center=[7.08, 8.88], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [ ]:
""" 



m = geemap.Map()
m.set_center(-112.8598, 36.2841, 10)
m.add_layer(slope, {'min': 0, 'max': 60}, 'slope')
m
"""

### . Land Cover / Existing Urban Layer (GLC_FCS30D: 2012–2022)

The raw GLC_FCS30D asset is stored as tiled, multi-band images (one band per
year). We mosaic the tiles, rename the bands to actual years, and convert the
multi-band image into a proper year-indexed `ImageCollection`, following the
official preprocessing pattern published by the data provider.

In the GLC_FCS30D legend, class code **190 = Impervious surfaces** (built-up /
urban). We use this to derive a binary urban mask for every year.


In [11]:
# GLC_FCS30D Global Land Cover
# Data source : https://gee-community-catalog.org/projects/glc_fcs/?h=glc+fcs30d
# Reference   : https://gee-community-catalog.org/tutorials/examples/glc_fcs30d_lulc/

# Annual land cover ImageCollection (2000 – 2022). 
# Each image in annual land cover data has 23 bands, one for each year from 2000-2022 (23 years)
# (bands b1, b2,..b23 = 2000, 2001,..2022)
glc_annual = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual")

# Classification scheme
# (35 landcover class and 1 fill value)
glc_class_values =  [
  10, 11, 12, 20, 51, 52, 61, 62, 71, 72, 81, 82, 91, 92, 120, 121, 122, 
  130, 140, 150, 152, 153, 181, 182, 183, 184, 185, 186, 187, 190, 200, 
  201, 202, 210, 220, 0
]

# Land cover class names
glc_class_names = [
    "Rainfed_cropland", "Herbaceous_cover_cropland", "Tree_or_shrub_cover_cropland",
    "Irrigated_cropland", "Open_evergreen_broadleaved_forest", "Closed_evergreen_broadleaved_forest",
    "Open_deciduous_broadleaved_forest", "Closed_deciduous_broadleaved_forest",
    "Open_evergreen_needle_leaved_forest", "Closed_evergreen_needle_leaved_forest",
    "Open_deciduous_needle_leaved_forest", "Closed_deciduous_needle_leaved_forest",
    "Open_mixed_leaf_forest", "Closed_mixed_leaf_forest", "Shrubland",
    "Evergreen_shrubland", "Deciduous_shrubland", "Grassland", "Lichens_and_mosses",
    "Sparse_vegetation", "Sparse_shrubland", "Sparse_herbaceous", "Swamp", "Marsh",
    "Flooded_flat", "Saline", "Mangrove", "Salt_marsh", "Tidal_flat",
    "Impervious_surfaces", "Bare_areas", "Consolidated_bare_areas",
    "Unconsolidated_bare_areas", "Water_body", "Permanent_ice_and_snow", "Filled_value",
]

glc_class_colours = [
    "#ffff64", "#ffff64", "#ffff00", "#aaf0f0", "#4c7300",
    "#006400", "#a8c800", "#00a000", "#005000", "#003c00",
    "#286400", "#285000", "#a0b432", "#788200", "#966400",
    "#964b00", "#966400", "#ffb432", "#ffdcd2", "#ffebaf",
    "#ffd278", "#ffebaf", "#00a884", "#73ffdf", "#9ebb3b",
    "#828282", "#f57ab6", "#66cdab", "#444f89", "#c31400",
    "#fff5d7", "#dcdcdc", "#fff5d7", "#0046c8", "#ffffff", "#ffffff",
]

In [12]:
# Mosaic tiled images and rename bands b1, b2, ... to 2000, 2001, ...
glc_mosaic = glc_annual.mosaic()
years_list = ee.List.sequence(2000, 2022).map(lambda year: ee.Number(year).format("%04d"))
glc_mosaic_renamed = glc_mosaic.rename(years_list)

# Multiband to single-band annual mosaic & assign time and year metadata to each 
glc_mosaic_renamed_upd = years_list.map(
    lambda year: glc_mosaic_renamed
    .select([year])
    .set({
        "system:time_start": ee.Date.fromYMD(ee.Number.parse(year), 1, 1).millis(),
        "system:index": year,
        "year": ee.Number.parse(year),
    })
)

glc_mosaics_col = ee.ImageCollection.fromImages(glc_mosaic_renamed_upd)
print(glc_mosaics_col.first().getInfo())

# Remap native class values to sequential integers
new_class_values = ee.List.sequence(1, ee.List(glc_class_values).length())

{'type': 'Image', 'bands': [{'id': '2000', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}], 'properties': {'system:time_start': 946684800000, 'year': 2000, 'system:index': '2000'}}


## Drivers of Urban Growth / Predictors

In [13]:
# Create maps to visualise thematic layers
thematic_map_1 = geemap.Map(center = fct_center, zoom=8)
thematic_map_1.add_basemap("SATELLITE")

thematic_map_2 = geemap.Map(center = fct_center, zoom=8)
thematic_map_2.add_basemap("SATELLITE")

**Elevation/DEM**

In [16]:
# Download elevation and compute slope
# Data source: https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003
dem = ee.Image('USGS/SRTMGL1_003')
print(f"Bands in the DEM {dem.bandNames().getInfo()}")


elevation = dem.select('elevation')
slope = ee.Terrain.slope(elevation)



thematic_map_1.addLayer(dem.clip(aoi), {'min': 0, 'max': 500, "palette": ["Red", "Green", "Yellow"]}, "DEM")
thematic_map_1.addLayer(slope.clip(aoi), {'min': 0, 'max': 10, "palette": ["Red", "Green", "Yellow"]}, "Slope")
thematic_map_1


Bands in the DEM ['elevation']


Map(bottom=31412.0, center=[9.056266, 7.498522], controls=(WidgetControl(options=['position', 'transparent_bg'…

**Distance to Road**

In [18]:
# Distance to roads
# Data source: https://gee-community-catalog.org/projects/grip/
roads_africa = ee.FeatureCollection("projects/sat-io/open-datasets/GRIP4/Africa")

# Roads that intersect with study extent
roads_aoi = roads_africa.filterBounds(aoi_bbox)

# Convert road from vector to raster & compute eucledian distance 
roads_raster = ee.Image().float().paint(roads_aoi, 1).clip(aoi)
distance_to_roads = (
    roads_raster.fastDistanceTransform(256)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())  # convert pixel distance to meters
    .rename("dist_to_roads")
    .clip(aoi)
)

# Maximum distance in 'distance_to_roads' layer
print(distance_to_roads.reduceRegion(ee.Reducer.max(), aoi, 1000, maxPixels=1e9).getInfo())

thematic_map_1.addLayer(roads_raster, vis_params_roads_raster, "Road Raster")
thematic_map_1.addLayer(distance_to_roads.select("dist_to_roads"), vis_params_dist_road, "Distance to Road")
thematic_map_1.addLayer(roads_aoi, vis_params_roads_vector, "Road Vector")
thematic_map_1

{'dist_to_roads': 21729.478200706984}


Map(bottom=31412.0, center=[9.058702156392139, 7.498168945312501], controls=(WidgetControl(options=['position'…

**Distance to Permanent Water**

In [ ]:
# Permanent water (JRC Global Surface Water)
# Data Source: https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_GlobalSurfaceWater
gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").clip(aoi)
permanent_water = gsw.select("occurrence").gte(50)  # >=50% of the time = water

# compute eucledian distance 
distance_to_water = (
    permanent_water.Not()
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt())
    .rename("dist_to_water")
    .clip(aoi)
)

# 
print(distance_to_water.reduceRegion(ee.Reducer.max(), aoi, 1000, maxPixels=1e9).getInfo())

thematic_map_1.addLayer(permanent_water, vis_params_water, "Permanent Water")
thematic_map_1.addLayer(distance_to_water.select("dist_to_water"), vis_params_dist_water, "Distance to Water")
thematic_map_1

{'dist_to_water': 38497.39218440647}


Map(bottom=62616.0, center=[8.809082353052137, 7.825012207031251], controls=(WidgetControl(options=['position'…

**Nighttime Lights**

In [28]:
date = ee.Date("2015-01-01")
date_plus_1year = date.advance(-1, "year")

print(date.format("YYYY-MM-dd").getInfo())
print(date_plus_1year.format("YYYY-MM-dd").getInfo())

2015-01-01
2014-01-01


In [29]:
# Nighttime lights (VIIRS DNB, annual composite)
# Data Source: https://developers.google.com/earth-engine/datasets/catalog/NOAA_VIIRS_DNB_MONTHLY_V1_VCMSLCFG
def get_nighttime_lights(year, study_extent):
    '''Annual mean VIIRS radiance composite.'''
    start = ee.Date.fromYMD(year, 1, 1) # "2015", "january", "1"
    end = start.advance(1, "year")
    composite = (
        ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
        .filterDate(start, end)
        .select("avg_rad")
        .mean()
        .rename("ntl")
        .clip(study_extent)
    )
    return composite


# Apply function 
ntl_2015 = get_nighttime_lights(2015, aoi)
ntl_2020 = get_nighttime_lights(2020, aoi)
ntl_2025 = get_nighttime_lights(2025, aoi)


thematic_map_2.addLayer(ntl_2015, vis_params_ntl, "NTL - 2015")
thematic_map_2.addLayer(ntl_2020, vis_params_ntl, "NTL - 2020")
thematic_map_2.addLayer(ntl_2025, vis_params_ntl, "NTL - 2025")
thematic_map_2

Map(center=[9.056266, 7.498522], controls=(WidgetControl(options=['position', 'transparent_bg'], position='top…

**Population Density**

In [37]:
# Population density (CIESIN GPWv4.11) 
# Data source: https://developers.google.com/earth-engine/datasets/catalog/CIESIN_GPWv411_GPW_Population_Density
# Closest available year to analysis baseline (2015 / 2020 / 2022).
gpw = ee.ImageCollection("CIESIN/GPWv411/GPW_Population_Density")

def get_population_density(year, extent):
    '''Return the GPWv4.11 population density image closest to the given year.'''
    available_years = [2000, 2005, 2010, 2015, 2020]
    closest_year = min(available_years, key=lambda y: abs(y - year))
    image = (
        gpw.filter(ee.Filter.calendarRange(closest_year, closest_year, "year"))
        .first()
        .select("population_density")
        .rename("pop_density")
        .clip(extent)
    )
    return image

# Apply function
pop_density_2015 = get_population_density(2015, aoi)
pop_density_2020 = get_population_density(2020, aoi)
pop_density_2022 = get_population_density(2022, aoi)   


thematic_map_2.addLayer(pop_density_2015, vis_params_gpw, "Population Density - 2015")
thematic_map_2.addLayer(pop_density_2020, vis_params_gpw, "Population Density - 2020")
thematic_map_2.addLayer(pop_density_2022, vis_params_gpw, "Population Density - 2022 (uses 2020 data)")

thematic_map_2

Map(bottom=62621.0, center=[8.795511185738743, 7.580566406250001], controls=(WidgetControl(options=['position'…

**Building Density**

In [ ]:
""" 



m = geemap.Map()
m.set_center(-112.8598, 36.2841, 10)
m.add_layer(slope, {'min': 0, 'max': 60}, 'slope')
m
"""

" \n\n\n\nm = geemap.Map()\nm.set_center(-112.8598, 36.2841, 10)\nm.add_layer(slope, {'min': 0, 'max': 60}, 'slope')\nm\n"

**Spectral Indices**

In [ ]:
# Spectral indices from Landsat (annual cloud-masked median composite) 
# Data Source: https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2
# Data Source: https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC09_C02_T1_L2
def mask_landsat_clouds(image):
    qa = image.select("QA_PIXEL")
    cloud_bit, shadow_bit = 1 << 3, 1 << 4
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(shadow_bit).eq(0))
    return image.updateMask(mask)

def get_spectral_indices(year):
    '''NDVI, NDBI, NDWI annual median composite from Landsat 8/9 SR.'''
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")

    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(aoi).filterDate(start, end)
    l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(aoi).filterDate(start, end)
    landsat = l8.merge(l9).map(mask_landsat_clouds)

    # Scale factors for Collection 2 Level-2 surface reflectance
    def apply_scale_factors(image):
        optical = image.select("SR_B.").multiply(0.0000275).add(-0.2)
        return image.addBands(optical, None, True)

    landsat = landsat.map(apply_scale_factors)
    composite = landsat.median().clip(aoi)

    ndvi = composite.normalizedDifference(["SR_B5", "SR_B4"]).rename("ndvi")
    ndbi = composite.normalizedDifference(["SR_B6", "SR_B5"]).rename("ndbi")
    ndwi = composite.normalizedDifference(["SR_B3", "SR_B5"]).rename("ndwi")

    return ee.Image.cat([ndvi, ndbi, ndwi])

indices_2015 = get_spectral_indices(2015)
indices_2020 = get_spectral_indices(2020)
indices_2025 = get_spectral_indices(2025)


# Add layers to map 
for year, indices in [(2015, indices_2015), (2020, indices_2020), (2025, indices_2025)]:
    thematic_map_2.addLayer(indices.select("ndvi"), ndvi_vis, f"NDVI - {year}")
    thematic_map_2.addLayer(indices.select("ndbi"), ndbi_vis, f"NDBI - {year}")
    thematic_map_2.addLayer(indices.select("ndwi"), ndwi_vis, f"NDWI - {year}")

thematic_map_2

Map(bottom=62694.0, center=[8.597315884206026, 7.814025878906251], controls=(WidgetControl(options=['position'…